<a href="https://colab.research.google.com/github/adhh0715/swiggy-order-quality-refund-analyzer/blob/main/Swiggy_Order_Quality_Refund_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Swiggy Order Quality, Wrong Orders & Refund Analytics**

## **Your Engineering Handover**

You have joined the Swiggy Order Quality team as a **Junior Python Developer**. The operations lead spends hours switching between food delivery records, refund requests, and customer complaints to identify late deliveries, wrong orders, restaurant quality issues, and cases requiring attention. You have been asked to build a reproducible **order quality analysis workflow** in this notebook.

Your deliverable is **not an app or dashboard**. It is a trustworthy set of cleaned data tables, an order-level investigation report, and a concise evidence-based handover to Order Quality Operations.

**How to work:** Make your own copy in Google Drive (`File → Save a copy in Drive`), upload the three provided files, complete the ten milestones in order, and run the assessment cell at the end. Write your own implementation in the code cells. Keep every required named output available for the next task and for assessment.

**Input files (only three):** `orders.csv` · `refund_requests.xlsx` · `customer_complaints.json`  
**Data:** Fictional training records; not actual Swiggy customer records.

> **Important:** The assessment at the end checks key output contracts and consistency. Passing it is not proof that every business calculation or written explanation is correct; reviewers may also inspect selected notebooks.

### **Important Note: Guidelines for Using AI Tools**

You may use AI tools such as ChatGPT to understand concepts, debug errors, or get help with specific code snippets and individual problems.

**Do not use AI to generate the entire project solution in one go.** Work through each milestone yourself, understand the Pandas operations you apply, and independently check your results.

The learning objective is practical problem-solving, debugging, and evidence-based data analysis.

## **Workspace setup · Data intake**
Extract the supplied dataset ZIP and upload its **three individual files** into the notebook runtime. Run setup first; no fixed external download link is required.

[Link to Dataset](https://drive.google.com/file/d/1frQy3LnmE-FyXLvgU6fWtcaHuunYW-D3/view?usp=sharing)

In [1]:
# Reproducible intake from the supplied AlmaBetter training-data ZIP.
from pathlib import Path
import pandas as pd
import numpy as np
import zipfile
from IPython.display import display, Markdown
DATA_DIR = Path('/content') if Path('/content').is_dir() else Path.cwd()
REQUIRED_FILES = ['orders.csv', 'refund_requests.xlsx', 'customer_complaints.json']
if any(not (DATA_DIR / name).is_file() for name in REQUIRED_FILES):
    import gdown
    archive = DATA_DIR / 'swiggy_project_datasets.zip'
    gdown.download('https://drive.google.com/file/d/1frQy3LnmE-FyXLvgU6fWtcaHuunYW-D3/view', str(archive), fuzzy=True, quiet=False)
    with zipfile.ZipFile(archive) as bundle:
        for name in REQUIRED_FILES:
            matches = [p for p in bundle.namelist() if Path(p).name == name]
            if len(matches) != 1:
                raise ValueError(f'Expected one {name}, found {len(matches)}')
            (DATA_DIR / name).write_bytes(bundle.read(matches[0]))
assert all((DATA_DIR / name).is_file() for name in REQUIRED_FILES)
print('All three supplied synthetic source files are present.')

All three supplied synthetic source files are present.


## Milestone 1 · Establish the order baseline
**From the team:** The Operations Lead needs a reliable first look at the food-delivery order feed. Load `orders.csv`, inspect schema, order/customer/restaurant counts, nulls and category distributions.

**Handover evidence:** `orders_df` plus a labeled quality and volume summary.

In [2]:
# Milestone 1 — inspect raw orders before making changes.
orders_df = pd.read_csv(DATA_DIR / 'orders.csv')
print('Raw rows:', len(orders_df))
print('Unique IDs:', orders_df[['order_id','customer_id','restaurant_id']].nunique().to_dict())
display(orders_df.head(), orders_df.tail())
orders_df.info()
display(orders_df.isna().sum().rename('missing_count').to_frame())
for column in ['order_status', 'delivery_city', 'cuisine_type', 'wrong_order_flag']:
    if column in orders_df:
        print(column)
        display(orders_df[column].value_counts(dropna=False).to_frame('records'))
print('Repeated raw order IDs:', orders_df.order_id.duplicated().sum())
print('Keep the raw feed unchanged; validate labels, IDs, dates and numeric fields before analysis.')

Raw rows: 5000
Unique IDs: {'order_id': 5000, 'customer_id': 1207, 'restaurant_id': 1012}


,order_id,customer_id,restaurant_id,order_date,order_time,order_amount,payment_mode,order_status,delivery_status,promised_delivery_mins,actual_delivery_mins,wrong_order_flag,cancellation_reason,delivery_city,device_type,app_version,cuisine_category
0,ORD004127,CUST00499,REST00702,2025-02-22,09:10:00,603.85,UPI,Delivered,Very Late,45,60.0,No,NaN,Pune,Android,9.3.0,Pizza
1,ORD002345,CUST00630,REST00253,2025-02-22,17:12:00,610.31,UPI,Delivered,On Time,45,60.0,No,NaN,Pune,Android,9.2.1,South Indian
2,ORD004612,CUST00202,REST00968,2025-02-14,13:19:00,927.41,Wallet,Delivered,On Time,40,39.0,No,NaN,Mumbai,Android,9.3.0,North Indian
3,ORD003731,CUST00014,REST00809,2025-01-02,16:52:00,726.34,Cash,Delivered,On Time,40,71.0,No,NaN,Chennai,Android,10.0.0,Chinese
4,ORD003938,CUST00428,REST00918,2025-02-16,21:34:00,964.43,Wallet,Cancelled,Failed,35,10.0,No,Customer cancelled,Mumbai,Android,10.0.0,Biryani


,order_id,customer_id,restaurant_id,order_date,order_time,order_amount,payment_mode,order_status,delivery_status,promised_delivery_mins,actual_delivery_mins,wrong_order_flag,cancellation_reason,delivery_city,device_type,app_version,cuisine_category
4995,ORD003096,CUST00381,REST00571,2025-04-08,16:24:00,215.81,Wallet,Delivered,On Time,35,41.0,No,NaN,Delhi NCR,Android,9.2.1,Burger
4996,ORD002897,CUST00363,REST00110,2025-02-17,19:20:00,956.51,Wallet,Delivered,Late,45,92.0,No,NaN,Delhi NCR,iOS,9.2.1,Desserts
4997,ORD000692,CUST00843,REST00313,2025-01-01,08:21:00,437.64,UPI,Delivered,On Time,45,70.0,No,NaN,Kolkata,Android,10.0.0,North Indian
4998,ORD003337,CUST00448,REST00517,2025-03-27,11:12:00,1227.64,NetBanking,Returned,Cancelled,45,70.0,No,NaN,Mumbai,Android,10.0.0,Healthy
4999,ORD001927,CUST01142,REST00071,2025-05-29,18:29:00,937.03,Cash,Delivered,On Time,25,69.0,No,NaN,Delhi NCR,iOS,10.0.0,Burger


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   order_id                5000 non-null   object 
 1   customer_id             5000 non-null   object 
 2   restaurant_id           5000 non-null   object 
 3   order_date              5000 non-null   object 
 4   order_time              5000 non-null   object 
 5   order_amount            5000 non-null   object 
 6   payment_mode            5000 non-null   object 
 7   order_status            5000 non-null   object 
 8   delivery_status         5000 non-null   object 
 9   promised_delivery_mins  5000 non-null   int64  
 10  actual_delivery_mins    4955 non-null   float64
 11  wrong_order_flag        5000 non-null   object 
 12  cancellation_reason     329 non-null    object 
 13  delivery_city           5000 non-null   object 
 14  device_type             5000 non-null   

,missing_count
order_id,0
customer_id,0
restaurant_id,0
order_date,0
order_time,0
order_amount,0
payment_mode,0
order_status,0
delivery_status,0
promised_delivery_mins,0


...

delivery_city


,records
delivery_city,
Bengaluru,1086
Mumbai,901
Delhi NCR,789
Hyderabad,652
Chennai,599
Pune,527
Kolkata,249
Ahmedabad,197


wrong_order_flag


,records
wrong_order_flag,
No,4588
Yes,412


Repeated raw order IDs: 0
Keep the raw feed unchanged; validate labels, IDs, dates and numeric fields before analysis.


## Milestone 2 · Onboard refunds and complaints
**From the team:** Finance shares `refund_requests.xlsx` and Customer Support shares `customer_complaints.json`. Inspect these sources, refund outcomes and complaint categories; document quality issues.

**Handover evidence:** raw `refunds_df` and `complaints_df` plus distributions.

In [3]:
# Milestone 2 — onboard related records without collapsing distinct requests.
refunds_df = pd.read_excel(DATA_DIR / 'refund_requests.xlsx')
complaints_df = pd.read_json(DATA_DIR / 'customer_complaints.json')
for name, frame, key in [('Refunds', refunds_df, 'refund_id'), ('Complaints', complaints_df, 'complaint_id')]:
    print(name, 'rows:', len(frame), 'columns:', list(frame.columns))
    display(frame.head())
    display(frame.isna().sum().to_frame('missing_count'))
    print('Repeated record IDs:', frame[key].duplicated().sum())
    for column in frame.select_dtypes(include=['object', 'string']).columns:
        if column.endswith('status') or column == 'complaint_type':
            display(frame[column].value_counts(dropna=False).to_frame(column))
print('Inspect spelling variants, blank IDs, invalid dates and currency-like text before joining.')

Refunds rows: 1300 columns: ['refund_id', 'order_id', 'refund_status', 'refund_amount', 'refund_requested_date', 'refund_completed_date', 'refund_reason', 'refund_channel']


,refund_id,order_id,refund_status,refund_amount,refund_requested_date,refund_completed_date,refund_reason,refund_channel
0,REF000001,ORD001754,Completed,341.85,2025-01-08,2025-01-11,Missing item,Call Center
1,REF000002,ORD002400,Requested,591.05,2025-01-29,NaN,Wrong order,Call Center
2,REF000003,ORD_ORPHAN_0001,Requested,206.81,2025-01-28,NaN,Late delivery,App
3,REF000004,ORD002089,Pending,244.43,2025-05-19,NaN,Late delivery,Chat
4,REF000005,ORD004433,Approved,787.55,2025-03-08,NaN,Missing item,Email


,missing_count
refund_id,0
order_id,0
refund_status,0
refund_amount,0
refund_requested_date,0
refund_completed_date,868
refund_reason,0
refund_channel,0


Repeated record IDs: 0


,refund_status
refund_status,
Completed,432
Pending,360
Approved,208
Requested,151
Failed,80
pending,40
rejected,29


...

,missing_count
complaint_id,0
order_id,0
customer_id,0
complaint_date,0
complaint_type,0
complaint_description,0
complaint_status,0
sentiment_tag,0


Repeated record IDs: 0


,complaint_type
complaint_type,
Cold food,195
Spillage,181
Missing item,174
Poor packaging,173
Late delivery,171
Rude delivery partner,166
Refund not received,163
Wrong order,162
POOR PACKAGING,18


,complaint_status
complaint_status,
Resolved,513
In Progress,372
Open,304
Escalated,175
closed,77
open,59


Inspect spelling variants, blank IDs, invalid dates and currency-like text before joining.


## Milestone 3 · Prepare reliable food orders
**From the team:** Clean duplicated order IDs, inconsistent labels, currency values, order dates and time-in-minutes fields. Preserve restaurant, cuisine and wrong-order indicators; do not treat missing actual time on cancelled orders as an on-time delivery.

**Handover evidence:** `orders_clean` one row per `order_id` and documented data-quality decisions.

In [4]:
# Milestone 3 — keep raw data, audit invalid values, never invent missing times.
def clean_text(values):
    return values.astype('string').str.strip().replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})

def clean_id(values):
    return clean_text(values).str.upper()

def nonnegative_number(values):
    parsed = pd.to_numeric(clean_text(values).str.replace(r'(?i)INR|Rs\.?|₹|,', '', regex=True).str.strip(), errors='coerce')
    return parsed.where(parsed.ge(0))

def clean_dates(values):
    return pd.to_datetime(clean_text(values), format='mixed', errors='coerce')

def deduplicate_ids(frame, key):
    # Prefer the most complete record; ties retain original source order.
    ranked = frame.assign(_completeness=frame.notna().sum(axis=1))
    return (ranked.sort_values('_completeness', ascending=False, kind='stable')
            .dropna(subset=[key]).drop_duplicates(key, keep='first')
            .drop(columns='_completeness').sort_index().reset_index(drop=True))

orders_prepared = orders_df.copy(deep=True)
required_order_cols = {'order_id','customer_id','restaurant_id','order_date','order_amount',
                       'order_status','delivery_city','promised_delivery_mins','actual_delivery_mins','wrong_order_flag'}
assert required_order_cols.issubset(orders_prepared.columns), 'Missing required order fields'
for col in orders_prepared.select_dtypes(include=['object','string']):
    orders_prepared[col] = clean_text(orders_prepared[col])
for col in ['order_id','customer_id','restaurant_id']:
    orders_prepared[col] = clean_id(orders_prepared[col])
for col in ['order_status','delivery_status','cuisine_category']:
    if col in orders_prepared:
        orders_prepared[col] = clean_text(orders_prepared[col]).str.title()
orders_prepared['delivery_city'] = clean_text(orders_prepared.delivery_city).str.title().replace({'Delhi Ncr':'Delhi NCR'})
orders_prepared['order_date'] = clean_dates(orders_prepared.order_date)
for col in ['order_amount','promised_delivery_mins','actual_delivery_mins']:
    orders_prepared[col] = nonnegative_number(orders_prepared[col])
orders_prepared['promised_delivery_mins'] = orders_prepared.promised_delivery_mins.where(orders_prepared.promised_delivery_mins.gt(0))
flag = clean_text(orders_prepared.wrong_order_flag).str.lower()
orders_prepared['wrong_order_flag'] = flag.map({'yes':'Yes','true':'Yes','1':'Yes','no':'No','false':'No','0':'No'}).astype('string')
orders_prepared['unknown_wrong_order_flag'] = orders_prepared.wrong_order_flag.isna()
orders_prepared['invalid_order_date'] = orders_prepared.order_date.isna()
orders_prepared['invalid_order_amount'] = orders_prepared.order_amount.isna()
orders_prepared['unrecognized_order_status'] = ~orders_prepared.order_status.isin(['Delivered','Cancelled','Returned','Failed'])
orders_prepared['missing_reference'] = orders_prepared[['customer_id','restaurant_id']].isna().any(axis=1)
order_duplicate_review = orders_prepared[orders_prepared.order_id.duplicated(keep=False)].copy()
orders_clean = deduplicate_ids(orders_prepared, 'order_id')
order_quality_summary = pd.DataFrame({'metric':['raw_rows','missing_order_id','duplicate_normalized_ids','clean_orders','invalid_date','invalid_amount','missing_actual_minutes'],
'value':[len(orders_df),int(orders_prepared.order_id.isna().sum()),int(orders_prepared.order_id.dropna().duplicated().sum()),len(orders_clean),int(orders_clean.invalid_order_date.sum()),int(orders_clean.invalid_order_amount.sum()),int(orders_clean.actual_delivery_mins.isna().sum())]})
display(order_quality_summary)
print('Missing cancellation reason on delivered orders is not imputed. Missing amounts/times remain unknown; invalid entries are retained for review. Duplicate selection uses completeness, not presumed recency.')
assert orders_clean.order_id.is_unique and orders_clean.order_id.notna().all()
assert len(orders_df) == len(orders_prepared)
display(orders_clean[['order_status','wrong_order_flag']].value_counts(dropna=False).to_frame('orders'))

,metric,value
0,raw_rows,5000
1,missing_order_id,0
2,duplicate_normalized_ids,0
3,clean_orders,5000
4,invalid_date,15
5,invalid_amount,0
6,missing_actual_minutes,45


Missing cancellation reason on delivered orders is not imputed. Missing amounts/times remain unknown; invalid entries are retained for review. Duplicate selection uses completeness, not presumed recency.


orders
order_status wrong_order_flag        
Delivered    No                  3852
             Yes                  351
Cancelled    No                   302
Returned     No                   234
Failed       No                   200
Cancelled    Yes                   27
Failed       Yes                   20
Returned     Yes                   14

## Milestone 4 · Clean refund and complaint histories
**From the team:** A single order can have multiple legitimate requests. Remove repeated **record IDs** only, standardize categorical labels, parse dates and numeric amounts; count missing/unknown order references without dropping legitimate issues.

**Handover evidence:** `refunds_clean`, `complaints_clean` and quality summaries.

In [5]:
# Milestone 4 — deduplicate by request ID, never by order ID.
refunds_prepared = refunds_df.copy(deep=True)
complaints_prepared = complaints_df.copy(deep=True)
for frame, key in [(refunds_prepared, 'refund_id'), (complaints_prepared, 'complaint_id')]:
    assert {key,'order_id'}.issubset(frame.columns)
    for col in frame.select_dtypes(include=['object','string']):
        frame[col] = clean_text(frame[col])
    for col in [key,'order_id','customer_id']:
        if col in frame:
            frame[col] = clean_id(frame[col])
    for col in frame.columns:
        if col.endswith('_date'):
            frame[col] = clean_dates(frame[col])
        elif col.endswith('_status') or col in ['complaint_type','refund_reason']:
            frame[col] = clean_text(frame[col]).str.title()
refunds_prepared['refund_amount'] = nonnegative_number(refunds_prepared.refund_amount)
refunds_clean = deduplicate_ids(refunds_prepared, 'refund_id')
complaints_clean = deduplicate_ids(complaints_prepared, 'complaint_id')
for frame in [refunds_clean, complaints_clean]:
    frame['unmatched_order_reference'] = ~frame.order_id.isin(orders_clean.order_id)
refunds_clean['invalid_refund_amount'] = refunds_clean.refund_amount.isna()
refunds_clean['invalid_requested_date'] = refunds_clean.refund_requested_date.isna()
refunds_clean['invalid_completion_sequence'] = (refunds_clean.refund_completed_date < refunds_clean.refund_requested_date).fillna(False)
refunds_clean['completed_without_date'] = refunds_clean.refund_status.eq('Completed') & refunds_clean.refund_completed_date.isna()
complaints_clean['invalid_complaint_date'] = complaints_clean.complaint_date.isna()
complaints_clean['customer_reference_mismatch'] = (complaints_clean.order_id.isin(orders_clean.order_id) &
    complaints_clean.customer_id.ne(complaints_clean.order_id.map(orders_clean.set_index('order_id').customer_id))).fillna(False)
refund_orphans = refunds_clean[refunds_clean.unmatched_order_reference].copy()
complaint_orphans = complaints_clean[complaints_clean.unmatched_order_reference].copy()
issue_quality_summary = pd.DataFrame([
 {'source':name,'raw_rows':len(raw),'missing_record_id':int(prep[key].isna().sum()),
  'duplicate_record_ids':int(prep[key].dropna().duplicated().sum()),'clean_rows':len(clean),
  'unmatched_orders':int(clean.unmatched_order_reference.sum())}
 for name,raw,prep,clean,key in [('refunds',refunds_df,refunds_prepared,refunds_clean,'refund_id'),
 ('complaints',complaints_df,complaints_prepared,complaints_clean,'complaint_id')]])
display(issue_quality_summary)
display(refunds_clean.refund_status.value_counts(dropna=False).to_frame('requests'),
        complaints_clean.complaint_status.value_counts(dropna=False).to_frame('complaints'))
print('Refund date/amount exceptions:', refunds_clean[['invalid_refund_amount','invalid_requested_date','invalid_completion_sequence','completed_without_date']].sum().to_dict())
print('Complaint date and customer-reference exceptions:', complaints_clean[['invalid_complaint_date','customer_reference_mismatch']].sum().to_dict())
print('Distinct requests are preserved; orphan records remain available in separate review tables. No unmatched ID is guessed.')

,source,raw_rows,missing_record_id,duplicate_record_ids,clean_rows,unmatched_orders
0,refunds,1300,0,0,1300,35
1,complaints,1500,0,0,1500,42


,requests
refund_status,
Completed,432
Pending,400
Approved,208
Requested,151
Failed,80
Rejected,29


,complaints
complaint_status,
Resolved,513
In Progress,372
Open,363
Escalated,175
Closed,77


Refund date/amount exceptions: {'invalid_refund_amount': 0, 'invalid_requested_date': 13, 'invalid_completion_sequence': 0, 'completed_without_date': 0}
Complaint date and customer-reference exceptions: {'invalid_complaint_date': 9, 'customer_reference_mismatch': 0}
Distinct requests are preserved; orphan records remain available in separate review tables. No unmatched ID is guessed.


## Milestone 5 · Measure delivery reliability and wrong orders
**From the team:** The Delivery Lead needs delay measured against the **promised minutes**, not calendar days. Analyze delivery outcomes, late-delivery cities and orders flagged as incorrect food orders. Missing actual times should remain unknown.

**Handover evidence:** numeric `delivery_delay_mins` in `orders_clean`, Boolean `wrong_order_issue`, and `city_delay_summary` with `delivery_city`, `delayed_order_count`.

In [6]:
# Milestone 5 — strict >15-minute threshold; only delivered, measurable orders.
measurable_delivery = (orders_clean.order_status.eq('Delivered') &
    orders_clean.actual_delivery_mins.notna() & orders_clean.promised_delivery_mins.notna())
orders_clean['delivery_delay_mins'] = (orders_clean.actual_delivery_mins - orders_clean.promised_delivery_mins).where(measurable_delivery)
orders_clean['delayed_over_15_mins'] = orders_clean.delivery_delay_mins.gt(15).fillna(False).astype(bool)
orders_clean['wrong_order_issue'] = orders_clean.wrong_order_flag.eq('Yes').fillna(False).astype(bool)
orders_clean['delivery_timing_review'] = (orders_clean.order_status.eq('Delivered') & ~measurable_delivery).astype(bool)
orders_clean['incomplete_delivery_information'] = orders_clean[['actual_delivery_mins','promised_delivery_mins']].isna().any(axis=1)
orders_clean['measurable_delivery'] = measurable_delivery.astype(bool)
city_delay_summary = orders_clean.groupby('delivery_city', dropna=False, as_index=False).agg(
    total_orders=('order_id','size'), measurable_deliveries=('measurable_delivery','sum'),
    delayed_order_count=('delayed_over_15_mins','sum'), timing_review_count=('delivery_timing_review','sum'),
    average_delay_mins=('delivery_delay_mins','mean'))
city_delay_summary['delay_rate_pct'] = (100*city_delay_summary.delayed_order_count / city_delay_summary.measurable_deliveries.replace(0,np.nan)).round(2)
city_delay_summary = city_delay_summary.sort_values('delayed_order_count', ascending=False)
delayed_orders = orders_clean[orders_clean.delayed_over_15_mins].copy()
display(orders_clean.order_status.value_counts().to_frame('orders'), city_delay_summary)
print('Delivered:', int(orders_clean.order_status.eq('Delivered').sum()), 'Measurable:', int(measurable_delivery.sum()),
      'Delayed >15:', len(delayed_orders), 'Timing review:', int(orders_clean.delivery_timing_review.sum()),
      'Wrong-order flags:', int(orders_clean.wrong_order_issue.sum()))
print('Timing labels are not used as a substitute for the duration calculation. Early deliveries can have negative delays.')
assert orders_clean.loc[~measurable_delivery,'delivery_delay_mins'].isna().all()
assert not orders_clean.loc[orders_clean.delivery_delay_mins.eq(15),'delayed_over_15_mins'].any()
assert not orders_clean.loc[orders_clean.order_status.ne('Delivered'),'delayed_over_15_mins'].any()

,orders
order_status,
Delivered,4203
Cancelled,329
Returned,248
Failed,220


,delivery_city,total_orders,measurable_deliveries,delayed_order_count,timing_review_count,average_delay_mins,delay_rate_pct
1,Bengaluru,1086,915,420,7,14.627322,45.90
6,Mumbai,901,758,379,5,16.005277,50.00
3,Delhi NCR,789,657,329,8,17.512938,50.08
4,Hyderabad,652,526,267,2,16.032319,50.76
2,Chennai,599,503,250,2,16.789264,49.70
7,Pune,527,438,214,8,17.575342,48.86
5,Kolkata,249,210,99,1,15.138095,47.14
0,Ahmedabad,197,161,80,2,15.906832,49.69


Delivered: 4203 Measurable: 4168 Delayed >15: 2038 Timing review: 35 Wrong-order flags: 412
Timing labels are not used as a substitute for the duration calculation. Early deliveries can have negative delays.


## Milestone 6 · Connect refund requests to orders
**From the team:** Finance wants visibility into refund volume, totals, pending/failed outcomes and request-to-completion time. Aggregate requests by order *before* merging so the base order count stays stable. Preserve orders without refunds.

**Handover evidence:** `refunds_by_order` with `refund_count`, `total_refund_amount`, `pending_or_failed_refund_count` and `orders_refunds_df`.

In [7]:
# Milestone 6 — aggregate first, then a validated left join.
refunds_clean['pending_or_failed'] = refunds_clean.refund_status.isin(['Pending','Failed'])
refunds_clean['pending_or_failed_amount'] = refunds_clean.refund_amount.where(refunds_clean.pending_or_failed, 0)
refunds_clean['refund_completion_days'] = (refunds_clean.refund_completed_date - refunds_clean.refund_requested_date).dt.total_seconds()/86400
refunds_clean['refund_completion_days'] = refunds_clean.refund_completion_days.where(
    refunds_clean.refund_status.eq('Completed') & refunds_clean.refund_completion_days.ge(0))
refunds_by_order = refunds_clean.dropna(subset=['order_id']).groupby('order_id',as_index=False).agg(
    refund_count=('refund_id','size'), total_refund_amount=('refund_amount',lambda x:x.sum(min_count=1)),
    pending_or_failed_refund_count=('pending_or_failed','sum'),
    pending_or_failed_refund_amount=('pending_or_failed_amount',lambda x:x.sum(min_count=1)),
    mean_refund_completion_days=('refund_completion_days','mean'),
    refund_statuses=('refund_status',lambda x:' | '.join(sorted(set(x.dropna())))))
orders_refunds_df = orders_clean.merge(refunds_by_order, on='order_id', how='left', validate='one_to_one')
for col in ['refund_count','pending_or_failed_refund_count']:
    orders_refunds_df[col] = orders_refunds_df[col].fillna(0).astype(int)
no_refunds = orders_refunds_df.refund_count.eq(0)
for col in ['total_refund_amount','pending_or_failed_refund_amount']:
    orders_refunds_df.loc[no_refunds,col] = 0.0
assert len(orders_refunds_df)==len(orders_clean) and orders_refunds_df.order_id.is_unique
assert orders_refunds_df.refund_count.sum()==int(refunds_clean.order_id.isin(orders_clean.order_id).sum())
print('All requests:',len(refunds_clean),'Mapped:',int(orders_refunds_df.refund_count.sum()),'Orphans:',len(refund_orphans))
print('Pending/failed requests:',int(refunds_clean.pending_or_failed.sum()),
      'Known pending/failed amount:',round(refunds_clean.pending_or_failed_amount.sum(),2))
print('Requested/Approved are reported separately; the specified priority rule flags only Pending or Failed.')
display(refunds_clean.groupby('refund_status').agg(requests=('refund_id','size'),amount=('refund_amount','sum'),mean_completion_days=('refund_completion_days','mean')))
display(refunds_by_order.sort_values('refund_count',ascending=False).head())

All requests: 1300 Mapped: 1265 Orphans: 35
Pending/failed requests: 480 Known pending/failed amount: 261856.77
Requested/Approved are reported separately; the specified priority rule flags only Pending or Failed.


,requests,amount,mean_completion_days
refund_status,,,
Approved,208,111039.2,NaN
Completed,432,242004.45,6.988263
Failed,80,42522.93,NaN
Pending,400,219333.84,NaN
Rejected,29,17623.48,NaN
Requested,151,83987.14,NaN


,order_id,refund_count,total_refund_amount,pending_or_failed_refund_count,pending_or_failed_refund_amount,mean_refund_completion_days,refund_statuses
1299,ORD_ORPHAN_0035,1,667.84,0,0.0,5.0,Completed
0,ORD000001,1,407.22,0,0.0,2.0,Completed
1,ORD000005,1,574.13,0,0.0,3.0,Completed
2,ORD000006,1,411.41,1,411.41,NaN,Pending
1283,ORD_ORPHAN_0019,1,555.3,1,555.3,NaN,Pending


## Milestone 7 · Find repeated food-quality complaints
**From the team:** Customer Support needs counts by complaint type and status, repeat complaints on the same order and repeat contacts by customer. Preserve separate valid complaint records.

**Handover evidence:** `complaints_by_order` with `complaint_count`, `open_complaint_count`; show repeat order/customer patterns.

In [8]:
# Milestone 7 — preserve distinct complaints, including unmatched references.
complaints_clean['open_issue'] = ~complaints_clean.complaint_status.isin(['Resolved','Closed'])
complaints_clean['wrong_order_complaint'] = complaints_clean.complaint_type.eq('Wrong Order').fillna(False)
complaints_by_order = complaints_clean.dropna(subset=['order_id']).groupby('order_id',as_index=False).agg(
    complaint_count=('complaint_id','size'), open_complaint_count=('open_issue','sum'),
    wrong_order_complaint_count=('wrong_order_complaint','sum'),
    complaint_types=('complaint_type',lambda x:' | '.join(sorted(set(x.dropna())))))
complaint_category_summary = complaints_clean.groupby('complaint_type',dropna=False).agg(
    complaint_count=('complaint_id','size'),unresolved_complaint_count=('open_issue','sum')).sort_values('complaint_count',ascending=False)
customer_complaint_summary = complaints_clean.groupby('customer_id',dropna=False).agg(
    complaint_count=('complaint_id','size'),unique_orders=('order_id','nunique'),
    unresolved_complaint_count=('open_issue','sum')).sort_values('complaint_count',ascending=False)
repeat_order_complaints = complaints_by_order[complaints_by_order.complaint_count.gt(1)].copy()
assert complaints_by_order.complaint_count.sum()==complaints_clean.order_id.notna().sum()
display(complaint_category_summary, complaints_clean.complaint_status.value_counts().to_frame('complaints'))
print('Orders referenced by multiple complaints:',len(repeat_order_complaints),
      'Customers with multiple contacts:',int(customer_complaint_summary.complaint_count.gt(1).sum()))
display(repeat_order_complaints.head(), customer_complaint_summary.head())
print('Wrong-order complaints are reported as supporting evidence. The priority signal uses the order feed flag, per the stated rule; it is not overwritten by complaint text.')

,complaint_count,unresolved_complaint_count
complaint_type,,
Cold Food,202,120
Spillage,193,117
Poor Packaging,191,116
Late Delivery,186,109
Missing Item,184,111
Rude Delivery Partner,184,113
Refund Not Received,181,112
Wrong Order,179,112


,complaints
complaint_status,
Resolved,513
In Progress,372
Open,363
Escalated,175
Closed,77


Orders referenced by multiple complaints: 175 Customers with multiple contacts: 387


,order_id,complaint_count,open_complaint_count,wrong_order_complaint_count,complaint_types
16,ORD000087,2,1,0,Late Delivery | Rude Delivery Partner
20,ORD000095,2,0,0,Refund Not Received
27,ORD000126,2,1,1,Spillage | Wrong Order
31,ORD000139,3,3,1,Late Delivery | Refund Not Received | Wrong Order
32,ORD000143,2,2,0,Cold Food | Rude Delivery Partner


,complaint_count,unique_orders,unresolved_complaint_count
customer_id,,,
CUST00183,9,8,3
CUST00431,8,4,5
CUST00630,8,6,4
CUST00136,6,5,5
CUST00609,6,5,3


Wrong-order complaints are reported as supporting evidence. The priority signal uses the order feed flag, per the stated rule; it is not overwritten by complaint text.


## Milestone 8 · Build an order-level quality investigation table
**From the team:** Support agents need one place to view food-delivery delays, wrong-order flags, refund issues and complaints. Join order-level aggregates using validated one-to-one merges, preserve all orders and report orphan issue records.

**Handover evidence:** `final_orders_df` with one row per clean order, business issue columns, and a join-reconciliation summary.

In [9]:
# Milestone 8 — one base order, one joined record.
final_orders_df = orders_refunds_df.merge(complaints_by_order,on='order_id',how='left',validate='one_to_one')
for col in ['complaint_count','open_complaint_count','wrong_order_complaint_count']:
    final_orders_df[col] = final_orders_df[col].fillna(0).astype(int)
final_orders_df['complaint_types'] = final_orders_df.complaint_types.fillna('None recorded')
final_orders_df['refund_statuses'] = final_orders_df.refund_statuses.fillna('None recorded')
reconciliation_summary = pd.DataFrame([
 {'source':'orders','clean_records':len(orders_clean),'mapped_records':len(final_orders_df),'unmatched_records':0},
 {'source':'refunds','clean_records':len(refunds_clean),'mapped_records':int(final_orders_df.refund_count.sum()),'unmatched_records':len(refund_orphans)},
 {'source':'complaints','clean_records':len(complaints_clean),'mapped_records':int(final_orders_df.complaint_count.sum()),'unmatched_records':len(complaint_orphans)}])
assert final_orders_df.order_id.is_unique and final_orders_df.order_id.notna().all()
assert set(final_orders_df.order_id)==set(orders_clean.order_id)
for source,col in [(refunds_clean,'refund_count'),(complaints_clean,'complaint_count')]:
    expected = source[source.order_id.isin(orders_clean.order_id)].groupby('order_id').size()
    actual = final_orders_df.set_index('order_id')[col]
    assert actual.eq(expected.reindex(actual.index,fill_value=0)).all()
assert (reconciliation_summary.clean_records == reconciliation_summary.mapped_records + reconciliation_summary.unmatched_records).all()
display(reconciliation_summary)
print('No-history orders retained:',int(final_orders_df.refund_count.eq(0).sum()),'without refunds;',
      int(final_orders_df.complaint_count.eq(0).sum()),'without complaints')
display(final_orders_df[['order_id','restaurant_id','delivery_delay_mins','wrong_order_issue','refund_count','complaint_count']].head())

,source,clean_records,mapped_records,unmatched_records
0,orders,5000,5000,0
1,refunds,1300,1265,35
2,complaints,1500,1458,42


No-history orders retained: 3735 without refunds; 3737 without complaints


,order_id,restaurant_id,delivery_delay_mins,wrong_order_issue,refund_count,complaint_count
0,ORD004127,REST00702,15.0,False,1,0
1,ORD002345,REST00253,15.0,False,0,0
2,ORD004612,REST00968,-1.0,False,1,0
3,ORD003731,REST00809,31.0,False,0,0
4,ORD003938,REST00918,<NA>,False,0,0


## Milestone 9 · Create a food-support triage queue
**From the team:** Use four transparent Swiggy-specific risk signals: **delivered more than 15 minutes late**, a **wrong order**, a **refund requiring follow-up**, and **multiple complaints**. Assign `High` for 2+ signals, `Medium` for exactly 1, and `Low` for none. Track missing delivered-order timing separately; do not infer that it is on time.

**Handover evidence:** Boolean signals `delayed_over_15_mins`, `wrong_order_issue`, `refund_needs_attention`, `repeated_complaints`, plus `support_priority` and review cases.

In [10]:
# Milestone 9 — transparent, reproducible support priority.
final_orders_df['delayed_over_15_mins'] = (final_orders_df.order_status.eq('Delivered') & final_orders_df.delivery_delay_mins.gt(15)).fillna(False).astype(bool)
final_orders_df['wrong_order_issue'] = final_orders_df.wrong_order_issue.fillna(False).astype(bool)
final_orders_df['refund_needs_attention'] = final_orders_df.pending_or_failed_refund_count.gt(0).astype(bool)
final_orders_df['repeated_complaints'] = final_orders_df.complaint_count.gt(1).astype(bool)
signal_columns = ['delayed_over_15_mins','wrong_order_issue','refund_needs_attention','repeated_complaints']
final_orders_df['issue_count'] = final_orders_df[signal_columns].sum(axis=1).astype(int)
def support_level(count):
    return 'High' if count >= 2 else ('Medium' if count == 1 else 'Low')
final_orders_df['support_priority'] = final_orders_df.issue_count.map(support_level)
final_orders_df['recommended_action'] = np.select(
 [final_orders_df.wrong_order_issue,final_orders_df.refund_needs_attention,final_orders_df.repeated_complaints,final_orders_df.delayed_over_15_mins],
 ['Restaurant Operations: investigate wrong-order evidence','Refund Operations: review Pending/Failed requests',
  'Customer Support: review repeated complaints','Delivery Operations: investigate delay'],default='Standard support process')
final_orders_df['manual_review_required'] = (final_orders_df.delivery_timing_review | final_orders_df.unknown_wrong_order_flag |
    final_orders_df.invalid_order_date | final_orders_df.invalid_order_amount | final_orders_df.missing_reference | final_orders_df.unrecognized_order_status)
manual_review_orders = final_orders_df[final_orders_df.manual_review_required].copy()
high_priority_orders = final_orders_df[final_orders_df.support_priority.eq('High')].copy()
# Boundary evidence: exactly 15 minutes is not >15; cancelled timings cannot count as completed delays.
boundary = pd.DataFrame({'status':['Delivered','Delivered','Cancelled','Delivered'],
                         'delay':[15.0,16.0,80.0,np.nan]})
assert (boundary.status.eq('Delivered') & boundary.delay.gt(15)).tolist()==[False,True,False,False]
assert [support_level(i) for i in range(5)]==['Low','Medium','High','High','High']
assert final_orders_df.support_priority.notna().all()
display(final_orders_df.support_priority.value_counts().reindex(['High','Medium','Low'],fill_value=0).to_frame('order_count'))
print('Delivered timing exceptions:',int(final_orders_df.delivery_timing_review.sum()),'All manual-review orders:',len(manual_review_orders))
for level in ['High','Medium','Low']:
    print(level,'examples')
    display(final_orders_df.loc[final_orders_df.support_priority.eq(level),['order_id']+signal_columns+['issue_count','support_priority']].head(2))
print('Low means no observed rule-based signals, not guaranteed problem-free. Missing information stays in the manual-review queue.')

,order_count
support_priority,
High,433
Medium,2196
Low,2371


Delivered timing exceptions: 35 All manual-review orders: 50
High examples


,order_id,delayed_over_15_mins,wrong_order_issue,refund_needs_attention,repeated_complaints,issue_count,support_priority
11,ORD000935,True,False,True,False,2,High
17,ORD002606,False,False,True,True,2,High


Medium examples


,order_id,delayed_over_15_mins,wrong_order_issue,refund_needs_attention,repeated_complaints,issue_count,support_priority
3,ORD003731,True,False,False,False,1,Medium
7,ORD003585,True,False,False,False,1,Medium


Low examples


,order_id,delayed_over_15_mins,wrong_order_issue,refund_needs_attention,repeated_complaints,issue_count,support_priority
0,ORD004127,False,False,False,False,0,Low
1,ORD002345,False,False,False,False,0,Low


Low means no observed rule-based signals, not guaranteed problem-free. Missing information stays in the manual-review queue.


## Milestone 10 · Export an operations handover
**From the team:** Provide a sorted, readable detail report plus a priority summary for the Support Lead, and quantify restaurant-level issue patterns without collapsing the order-level deliverable. Explain 3–5 findings with real computed values and one data-quality caveat.

**Handover evidence:** `final_report`, `priority_summary`, `restaurant_issue_summary`; export `swiggy_order_quality_report.csv` and `swiggy_issue_summary.xlsx`.

In [11]:
# Milestone 10 — export the reports from the Pandas workflow required by the PRD.
priority_order = {'High':0,'Medium':1,'Low':2}
final_report = (final_orders_df.assign(_priority_rank=final_orders_df.support_priority.map(priority_order))
    .sort_values(['_priority_rank','issue_count','order_id'],ascending=[True,False,True],kind='stable')
    .drop(columns='_priority_rank').reset_index(drop=True))
priority_summary = (final_report.groupby('support_priority').size().reindex(['High','Medium','Low'],fill_value=0)
    .rename('order_count').reset_index())
priority_summary['share_pct'] = (100*priority_summary.order_count/len(final_report)).round(2)
restaurant_issue_summary = final_report.groupby('restaurant_id',dropna=False,as_index=False).agg(
    order_count=('order_id','size'), wrong_order_count=('wrong_order_issue','sum'),
    delayed_order_count=('delayed_over_15_mins','sum'), complaint_count=('complaint_count','sum'),
    unresolved_complaint_count=('open_complaint_count','sum'), refund_count=('refund_count','sum'),
    pending_or_failed_refund_count=('pending_or_failed_refund_count','sum'),
    high_priority_order_count=('support_priority',lambda x:x.eq('High').sum()))
restaurant_issue_summary['wrong_order_rate_pct'] = (100*restaurant_issue_summary.wrong_order_count/restaurant_issue_summary.order_count).round(2)
restaurant_issue_summary = restaurant_issue_summary.sort_values(['wrong_order_count','order_count'],ascending=False)
assert len(final_report)==len(orders_clean) and final_report.order_id.is_unique
assert priority_summary.order_count.sum()==len(final_report)
assert restaurant_issue_summary.order_count.sum()==len(final_report)
assert restaurant_issue_summary.wrong_order_count.sum()==int(final_report.wrong_order_issue.sum())
csv_path = DATA_DIR/'swiggy_order_quality_report.csv'
xlsx_path = DATA_DIR/'swiggy_issue_summary.xlsx'
final_report.to_csv(csv_path,index=False)
report_sheets = {'Priority Summary':priority_summary, 'Restaurant Quality':restaurant_issue_summary,
 'City Delays':city_delay_summary, 'Reconciliation':reconciliation_summary,
 'Order Data Quality':order_quality_summary,'Issue Data Quality':issue_quality_summary,
 'Refund Orphans':refund_orphans,'Complaint Orphans':complaint_orphans,
 'Manual Review':manual_review_orders}
with pd.ExcelWriter(xlsx_path,engine='openpyxl') as writer:
    for sheet_name,frame in report_sheets.items():
        frame.to_excel(writer,sheet_name=sheet_name,index=False)
        ws=writer.sheets[sheet_name]
        ws.freeze_panes='A2'
        ws.auto_filter.ref=ws.dimensions
        from openpyxl.styles import Font, PatternFill, Alignment
        from openpyxl.utils import get_column_letter
        for cell in ws[1]:
            cell.font=Font(bold=True,color='FFFFFF')
            cell.fill=PatternFill('solid',fgColor='253B53')
            cell.alignment=Alignment(wrap_text=True,vertical='center')
        ws.row_dimensions[1].height=32
        for i,column in enumerate(frame.columns,1):
            max_len=max(len(str(column)),*(len(str(v)) for v in frame[column].head(200).astype('string').fillna('')))
            ws.column_dimensions[get_column_letter(i)].width=min(max(14,max_len+2),42)
            for cell in list(ws.columns)[i-1][1:]:
                if 'pct' in column:
                    cell.number_format='0.00'
                elif any(t in column for t in ['amount','mins','days']):
                    cell.number_format='#,##0.00'
# Read exported files back and compare totals, not merely file existence.
assert len(pd.read_csv(csv_path))==len(final_report)
assert pd.read_excel(xlsx_path,sheet_name='Priority Summary').order_count.sum()==len(final_report)
assert pd.read_excel(xlsx_path,sheet_name='Restaurant Quality').order_count.sum()==len(final_report)
display(priority_summary,city_delay_summary,restaurant_issue_summary.head(10))
print('Verified exports:',csv_path.name, csv_path.stat().st_size,'bytes;',xlsx_path.name,xlsx_path.stat().st_size,'bytes')
late_count=int(final_report.delayed_over_15_mins.sum())
measured_count=int(final_report.measurable_delivery.sum())
high_count=int(final_report.support_priority.eq('High').sum())
top_city=city_delay_summary.iloc[0]
top_restaurant=restaurant_issue_summary.iloc[0]
findings=[
 f'{late_count:,} of {measured_count:,} measurable delivered orders ({100*late_count/measured_count:.2f}%) exceeded the promise by more than 15 minutes; investigate delivery operations rather than rely on status labels.',
 f'{high_count:,} of {len(final_report):,} orders meet at least two issue conditions and belong in the High-priority queue; coordinate restaurant, refund and customer-support follow-up.',
 f'{top_city.delivery_city} has the largest delayed-order count ({int(top_city.delayed_order_count):,}); compare its {top_city.delay_rate_pct:.2f}% rate with other cities before allocating resources.',
 f'Restaurant {top_restaurant.restaurant_id} has {int(top_restaurant.wrong_order_count)} flagged wrong orders among {int(top_restaurant.order_count)} orders ({top_restaurant.wrong_order_rate_pct:.2f}%); inspect its cases, but avoid ranking small samples as conclusive.',
 f'Data caveat: {int(final_report.delivery_timing_review.sum())} delivered orders have incomplete timing; {len(refund_orphans)} refund and {len(complaint_orphans)} complaint records have unmatched order IDs. All data is synthetic and the priority score reflects observed signals only.'
]
display(Markdown('### Computed business findings\n'+'\n'.join('- '+item for item in findings)))

,support_priority,order_count,share_pct
0,High,433,8.66
1,Medium,2196,43.92
2,Low,2371,47.42


,delivery_city,total_orders,measurable_deliveries,delayed_order_count,timing_review_count,average_delay_mins,delay_rate_pct
1,Bengaluru,1086,915,420,7,14.627322,45.90
6,Mumbai,901,758,379,5,16.005277,50.00
3,Delhi NCR,789,657,329,8,17.512938,50.08
4,Hyderabad,652,526,267,2,16.032319,50.76
2,Chennai,599,503,250,2,16.789264,49.70
7,Pune,527,438,214,8,17.575342,48.86
5,Kolkata,249,210,99,1,15.138095,47.14
0,Ahmedabad,197,161,80,2,15.906832,49.69


,restaurant_id,order_count,wrong_order_count,delayed_order_count,complaint_count,unresolved_complaint_count,refund_count,pending_or_failed_refund_count,high_priority_order_count,wrong_order_rate_pct
299,REST00302,9,4,5,3,2,2,0,2,44.44
560,REST00567,9,3,6,1,1,2,1,3,33.33
421,REST00425,8,3,4,2,2,1,1,2,37.50
923,REST00930,8,3,3,3,2,0,0,2,37.50
416,REST00420,7,3,2,1,1,3,0,0,42.86
884,REST00891,7,3,2,0,0,3,1,2,42.86
491,REST00497,5,3,2,0,0,0,0,1,60.00
300,REST00303,11,2,4,2,2,4,1,1,18.18
699,REST00706,11,2,3,6,5,4,1,2,18.18
854,REST00861,11,2,5,3,1,4,1,1,18.18


Verified exports: swiggy_order_quality_report.csv 1474283 bytes; swiggy_issue_summary.xlsx 62046 bytes


### Computed business findings
- 2,038 of 4,168 measurable delivered orders (48.90%) exceeded the promise by more than 15 minutes; investigate delivery operations rather than rely on status labels.
- 433 of 5,000 orders meet at least two issue conditions and belong in the High-priority queue; coordinate restaurant, refund and customer-support follow-up.
- Bengaluru has the largest delayed-order count (420); compare its 45.90% rate with other cities before allocating resources.
- Restaurant REST00302 has 4 flagged wrong orders among 9 orders (44.44%); inspect its cases, but avoid ranking small samples as conclusive.
- Data caveat: 35 delivered orders have incomplete timing; 35 refund and 42 complaint records have unmatched order IDs. All data is synthetic and the priority score reflects observed signals only.

In [12]:
# Operations controls — safe filters on the generated report, plus report downloads.
import ipywidgets as widgets
from IPython.display import clear_output
from google.colab import files

def filter_orders(query='', priority='All', city='All'):
    mask = pd.Series(True,index=final_report.index)
    if query.strip():
        term = query.strip().upper()
        mask &= final_report.order_id.str.contains(term,regex=False,na=False) | final_report.restaurant_id.str.contains(term,regex=False,na=False)
    if priority != 'All':
        mask &= final_report.support_priority.eq(priority)
    if city != 'All':
        mask &= final_report.delivery_city.eq(city)
    return final_report.loc[mask].copy()

search_box=widgets.Text(description='Order / restaurant:',style={'description_width':'initial'},placeholder='e.g. ORD004127')
priority_filter=widgets.Dropdown(options=['All','High','Medium','Low'],description='Priority:')
city_filter=widgets.Dropdown(options=['All']+sorted(final_report.delivery_city.dropna().unique()),description='City:')
search_button=widgets.Button(description='Search / filter',button_style='primary')
results_area=widgets.Output()

def show_filtered(_=None):
    with results_area:
        clear_output(wait=True)
        result=filter_orders(search_box.value,priority_filter.value,city_filter.value)
        print(f'{len(result):,} matching orders. Showing up to 20; exports contain all orders.')
        if result.empty:
            print('No matches. Check the identifier or clear the filters.')
        else:
            display(result[['order_id','restaurant_id','delivery_city','delivery_delay_mins','wrong_order_issue','refund_count','complaint_count','support_priority','manual_review_required']].head(20))
search_button.on_click(show_filtered)
# A literal query cannot run code or become a regular expression.
assert filter_orders('UNKNOWN-ID-XYZ').empty
assert len(filter_orders())==len(final_report)
assert filter_orders(priority='High').support_priority.eq('High').all()
assert filter_orders(final_report.order_id.iloc[0]).order_id.eq(final_report.order_id.iloc[0]).all()
csv_download=widgets.Button(description='Download detail CSV')
xlsx_download=widgets.Button(description='Download Excel summary')
csv_download.on_click(lambda _:files.download(str(csv_path)))
xlsx_download.on_click(lambda _:files.download(str(xlsx_path)))
display(Markdown('### Order investigation controls\nChoose filters, then select **Search / filter**. Missing timing remains a manual-review issue.'))
display(search_box,widgets.HBox([priority_filter,city_filter]),search_button,results_area,widgets.HBox([csv_download,xlsx_download]))
show_filtered()

### Order investigation controls
Choose filters, then select **Search / filter**. Missing timing remains a manual-review issue.

Text(value='', description='Order / restaurant:', placeholder='e.g. ORD004127', style=DescriptionStyle(descrip…

Button(button_style='primary', description='Search / filter', style=ButtonStyle())

Output()

### Operations handover — Aditya Siddaarth

**Project summary.** A reproducible Python/Pandas workflow combines the three supplied synthetic Swiggy feeds into one record per order, with CSV detail, Excel summaries, exception tables, support priority and notebook search controls. It does not process refunds or contact customers.

**GitHub repository:** [swiggy-order-quality-refund-analyzer](https://github.com/adhh0715/swiggy-order-quality-refund-analyzer)

**Computed findings:**
- 2,038 of 4,168 measurable delivered orders (48.90%) exceeded the promise by more than 15 minutes. Use the duration calculation rather than the inconsistent raw delivery labels.
- 433 of 5,000 orders (8.66%) are High priority; 2,196 (43.92%) are Medium and 2,371 (47.42%) Low. Begin coordinated follow-up with the High-priority queue.
- Bengaluru has the largest delayed-order count (420), at a 45.90% rate. Hyderabad has a higher rate (50.76%) despite a lower delayed count (267); use both volume and rate for planning.
- REST00302 has four wrong-order flags among nine orders (44.44%). Review individual cases; a small sample does not establish persistent restaurant performance.
- 35 delivered orders have unavailable timing. There are 35 unmatched refunds and 42 unmatched complaints. Retain these for review. All data is synthetic; Low means no observed rule-based signals, not guaranteed absence of problems.

**Method and decisions.** Preserve raw DataFrames. Normalize IDs and statuses; parse currency and dates. Duplicate record IDs are resolved by completeness with stable source-order ties, without collapsing distinct requests for an order. Invalid values remain missing and are audited. Missing cancellation reasons on delivered orders are not invented. Never infer missing actual times or guess relationships. Aggregate refund and complaint histories before validated one-to-one left joins. Only Pending/Failed refunds trigger the prescribed refund signal. Wrong-order complaint text remains supporting evidence; the order flag drives the specified signal.

**Validation.** All 5,000 orders are retained. Reconcile 1,265 matched refunds plus 35 unmatched requests and 1,458 matched complaints plus 42 unmatched records. Boundary checks cover exactly 15 minutes, non-delivered orders, missing timing, each priority level and invalid searches. Exported CSV/Excel totals are read back and checked. The unchanged automated assessment passed 100/100 after a fresh-session run; this is not a final human grade.

**Fix log.** Excel column-width formatting initially attempted to replace missing nullable numeric values with text. Convert the display-width sample to strings first, preserving the underlying numeric values.

**Re-run.** Restart the session and run all cells. Setup retrieves and extracts only the three supplied training files when absent; upload them manually if source access changes. Search controls and export-download buttons appear after Milestone 10.

**Presentation video:** Pending the learner's own recording. Use the guide below, rehearse the explanation, and add the real video link before submission.

## Recording guide — aim for 18–20 minutes

Use these as prompts while showing the notebook; explain the work in your own words. Rehearse the duration before recording.

### 0:00–2:00 — Problem and users
Introduce yourself and the project. Explain that support staff otherwise switch between orders, refunds and complaints. Show the three source filenames and describe the required deliverable: an investigation report, not a live refund application. Identify Customer Support, Delivery Operations, Refund Operations and Restaurant Operations as users. Say explicitly that the records are synthetic.

### 2:00–4:00 — Data intake and baseline
Show setup and Milestones 1–2. Explain CSV, Excel and JSON and the shared order_id key. Mention 5,000 orders, 1,300 refund requests and 1,500 complaints. Show the schema and missing-value outputs. Point out that raw DataFrames are kept so cleaning can be audited. The reproducible setup fetches the supplied archive; manual upload remains possible if the source becomes unavailable.

### 4:00–7:00 — Cleaning and quality decisions
Show Milestones 3–4 and explain strip, title-case statuses, currency parsing, date coercion and record-ID deduplication. Distinguish duplicate request IDs from several legitimate requests for one order. Explain why missing actual times cannot become zero, and why a missing cancellation reason on a delivered order can be legitimate. Show the 15 invalid order dates and the separate orphan tables: 35 refunds and 42 complaints. Explain that missing/invalid data is retained for review rather than guessed. Show the fix-log note about nullable numeric values and export formatting.

### 7:00–9:30 — Delivery, refunds and complaints
Show Milestones 5–7. Explain actual minus promised minutes for Delivered orders with both durations. Exactly 15 minutes is not greater than 15. Cancelled and unknown-time orders cannot be counted as completed delayed deliveries. Show 2,038 late orders among 4,168 measurable deliveries, and 35 delivered timing exceptions. Explain that Pending/Failed refunds trigger the specified refund condition; other statuses remain visible. Describe complaint counts and unresolved statuses.

### 9:30–12:00 — Joins and priority
Show Milestones 8–9 and the reconciliation table. Explain why joining raw one-to-many histories can multiply rows and money. First group each history by order_id, then use validated one-to-one left joins. Show all 5,000 orders retained, 1,265 matched refunds and 1,458 matched complaints. Walk through the four independent Boolean signals: delay >15, wrong order, pending/failed refund and more than one complaint. Explain High=two or more, Medium=one and Low=zero. Show one example from each priority. Explain why Low does not mean the underlying data is complete.

### 12:00–14:30 — Reports and findings
Show Milestone 10. Explain the CSV detail and Excel summary sheets, sorted priorities, restaurant summary, city delay rates and exception views. Mention 433 High-priority orders (8.66%). Compare Bengaluru volume (420 late) with Hyderabad rate (50.76%). Discuss REST00302, four wrong flags in nine orders, and explain the small-sample limitation. Recommend targeted case investigation rather than unsupported conclusions about a real business.

### 14:30–16:30 — Demonstrate controls and testing
Use the order/restaurant search, select High priority and a city, then clear filters. Try UNKNOWN-ID-XYZ and show the friendly no-match response. Explain literal matching and that the view shows up to 20 results while exports contain all orders. Show the download buttons. Show boundary assertions, validated merges, source reconciliation and the exported-file readback checks. Show the unchanged 100/100 automated scorecard and explain that it covers selected contracts rather than every business interpretation.

### 16:30–18:30 — Limitations, reuse and wrap-up
Explain synthetic data, invalid dates, missing timing, orphan relationships and small restaurant samples. Describe how a larger dataset could use chunking or database aggregation and how production would add access controls, monitoring and formal data contracts. Reflect on the operations you personally understand and any remaining questions. Show the repository link once available. Close by summarizing how the report directs support attention without automating refunds.

### Before recording
Run the notebook, keep outputs visible, test microphone/screen capture, avoid reading every line of code, and check that the recording lasts 15–25 minutes. Upload your own recording to an accessible location and use its actual link in the submission.


## Final milestone · Automated assessment
Run all cells in a **fresh kernel** before assessment; stale variables may create misleading passes. This 100-point scorecard checks structural and selected business rules, not every interpretation. Do not edit this cell for student submission.

In [13]:

# AUTOMATED MILESTONE ASSESSMENT — Run only after finishing Milestones 1–10.
# This is a non-secure learning aid; see the project brief for review expectations.
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

_scope = globals()
_checks = []

def check(task, criterion, fn):
    try:
        passed = bool(fn())
        msg = 'Meets check' if passed else 'Missing or inconsistent output'
    except Exception as ex:
        passed = False
        msg = f'Cannot verify: {type(ex).__name__}'
    _checks.append({'Milestone': task, 'Check': criterion, 'Result': 'Pass' if passed else 'Review', 'Points': 5 if passed else 0, 'Feedback': msg})

def df(name):
    obj = _scope.get(name)
    if not isinstance(obj, pd.DataFrame):
        raise ValueError(f'{name} is not a DataFrame')
    return obj

def has_cols(name, cols):
    return set(cols).issubset(df(name).columns)

def unique_nonnull(frame, key):
    return key in frame and frame[key].notna().all() and frame[key].is_unique

def numeric(frame, col):
    return col in frame and pd.api.types.is_numeric_dtype(frame[col])

def real_dates(frame, cols):
    return all(col in frame and pd.api.types.is_datetime64_any_dtype(frame[col]) for col in cols)

def normalized(frame, col):
    vals = frame[col].dropna().astype(str)
    return not vals.empty and vals.eq(vals.str.strip()).all() and vals.eq(vals.str.title()).all()

def issues_match():
    o = df('final_orders_df')
    r = df('refunds_clean')
    c = df('complaints_clean')
    for src, col in [(r, 'refund_count'), (c, 'complaint_count')]:
        counts = src[src.order_id.isin(o.order_id)].groupby('order_id').size()
        actual = o.set_index('order_id')[col].fillna(0)
        aligned = counts.reindex(actual.index, fill_value=0)
        if not actual.eq(aligned).all(): return False
    return True

def delivery_delay_matches():
    o = df('orders_clean')
    required = ['order_status', 'promised_delivery_mins', 'actual_delivery_mins', 'delivery_delay_mins']
    if not set(required).issubset(o.columns): return False
    if not all(numeric(o, col) for col in required[1:]): return False

    delivered = o.order_status.astype('string').str.strip().str.casefold().eq('delivered').fillna(False)
    valid = delivered & o.actual_delivery_mins.notna() & o.promised_delivery_mins.notna()
    if not valid.any(): return False

    expected = o.loc[valid, 'actual_delivery_mins'] - o.loc[valid, 'promised_delivery_mins']
    actual = o.loc[valid, 'delivery_delay_mins']

    return (actual.notna().all()
            and np.isclose(actual.to_numpy(dtype=float), expected.to_numpy(dtype=float), atol=1e-6).all()
            and o.loc[~valid, 'delivery_delay_mins'].isna().all())

def flags_match():
    o = df('final_orders_df')
    required = ['delayed_over_15_mins', 'wrong_order_issue', 'refund_needs_attention', 'repeated_complaints']
    if not set(required).issubset(o.columns): return False
    if any(not pd.api.types.is_bool_dtype(o[x]) for x in required): return False

    delivered = o.order_status.astype('string').str.strip().str.casefold().eq('delivered')
    delay_expected = (delivered & o.delivery_delay_mins.gt(15)).fillna(False)
    wrong_expected = o.wrong_order_flag.astype('string').str.strip().str.casefold().eq('yes').fillna(False)
    refund_expected = o.pending_or_failed_refund_count.fillna(0).gt(0)
    repeat_expected = o.complaint_count.fillna(0).gt(1)

    return all(o[col].fillna(False).eq(expected).all() for col, expected in zip(
        required, [delay_expected, wrong_expected, refund_expected, repeat_expected]
    ))

def priorities_match():
    o = df('final_orders_df')
    flags = ['delayed_over_15_mins', 'wrong_order_issue', 'refund_needs_attention', 'repeated_complaints']
    if not has_cols('final_orders_df', flags + ['support_priority']): return False
    if any(not pd.api.types.is_bool_dtype(o[x]) for x in flags): return False
    signals = o[flags].sum(axis=1)
    expected = signals.map(lambda x: 'High' if x >= 2 else ('Medium' if x == 1 else 'Low'))
    return o.support_priority.eq(expected).all()

def report_matches():
    r = df('final_report')
    o = df('final_orders_df')
    return (has_cols('final_report', ['order_id', 'support_priority']) and
            len(r) == len(o) and r.order_id.notna().all() and r.order_id.is_unique and
            set(r.order_id) == set(o.order_id) and
            r.set_index('order_id').support_priority.eq(
                o.set_index('order_id').loc[r.order_id, 'support_priority'].to_numpy()
            ).all())

def summary_matches():
    s = df('priority_summary')
    if not has_cols('priority_summary', ['support_priority', 'order_count']): return False
    expected = df('final_orders_df').support_priority.value_counts()
    got = s.groupby('support_priority').order_count.sum()
    return got.reindex(expected.index, fill_value=0).eq(expected).all() and got.sum() == len(df('final_orders_df'))

def restaurant_summary_matches():
    s = df('restaurant_issue_summary')
    o = df('final_orders_df')
    if not has_cols('restaurant_issue_summary', ['restaurant_id', 'order_count']): return False
    if not numeric(s, 'order_count'): return False
    expected = o.groupby('restaurant_id', dropna=False).size()
    actual = s.groupby('restaurant_id', dropna=False).order_count.sum()
    return actual.reindex(expected.index, fill_value=0).eq(expected).all() and actual.sum() == len(o)

check(1, 'Orders loaded, key fields present', lambda: has_cols('orders_df', ['order_id', 'customer_id', 'restaurant_id', 'order_status', 'order_amount']))
check(1, 'Orders source is non-empty', lambda: len(df('orders_df')) > 0)

check(2, 'Refund source loaded', lambda: has_cols('refunds_df', ['refund_id', 'order_id', 'refund_status', 'refund_amount']) and len(df('refunds_df')) > 0)
check(2, 'Complaint source loaded', lambda: has_cols('complaints_df', ['complaint_id', 'order_id', 'complaint_type', 'complaint_status']) and len(df('complaints_df')) > 0)

check(3, 'One clean record per order', lambda: unique_nonnull(df('orders_clean'), 'order_id') and len(df('orders_clean')) <= len(df('orders_df')))
check(3, 'Amounts, dates, statuses standardized', lambda: numeric(df('orders_clean'), 'order_amount') and real_dates(df('orders_clean'), ['order_date']) and normalized(df('orders_clean'), 'order_status'))

check(4, 'Refund records cleaned', lambda: unique_nonnull(df('refunds_clean'), 'refund_id') and numeric(df('refunds_clean'), 'refund_amount') and real_dates(df('refunds_clean'), ['refund_requested_date']) and normalized(df('refunds_clean'), 'refund_status'))
check(4, 'Complaint records cleaned', lambda: unique_nonnull(df('complaints_clean'), 'complaint_id') and real_dates(df('complaints_clean'), ['complaint_date']) and normalized(df('complaints_clean'), 'complaint_status'))

check(5, 'Delivery delay calculation is consistent', delivery_delay_matches)
check(5, 'City delay summary is available', lambda: has_cols('city_delay_summary', ['delivery_city', 'delayed_order_count']) and numeric(df('city_delay_summary'), 'delayed_order_count'))

check(6, 'Refund summary uses unique order IDs', lambda: has_cols('refunds_by_order', ['order_id', 'refund_count', 'total_refund_amount', 'pending_or_failed_refund_count']) and unique_nonnull(df('refunds_by_order'), 'order_id'))
check(6, 'Refund join keeps order population', lambda: unique_nonnull(df('orders_refunds_df'), 'order_id') and set(df('orders_refunds_df').order_id) == set(df('orders_clean').order_id))

check(7, 'Complaints summarized by order', lambda: has_cols('complaints_by_order', ['order_id', 'complaint_count', 'open_complaint_count']) and unique_nonnull(df('complaints_by_order'), 'order_id'))
check(7, 'Complaint totals match cleaned source', lambda: df('complaints_by_order').complaint_count.sum() == df('complaints_clean').order_id.notna().sum())

check(8, 'Master table contains all orders once', lambda: unique_nonnull(df('final_orders_df'), 'order_id') and set(df('final_orders_df').order_id) == set(df('orders_clean').order_id))
check(8, 'Master issue counts reconcile', issues_match)

check(9, 'Support signals follow business rules', flags_match)
check(9, 'Priority mapping matches signals', priorities_match)

check(10, 'Detailed report and export exist', lambda: report_matches() and (Path(DATA_DIR) / 'swiggy_order_quality_report.csv').is_file())
check(10, 'Priority and restaurant summaries reconcile; export exists', lambda: summary_matches() and restaurant_summary_matches() and (Path(DATA_DIR) / 'swiggy_issue_summary.xlsx').is_file())

assessment_df = pd.DataFrame(_checks)
score_by_task = assessment_df.groupby('Milestone', as_index=False).agg(Points=('Points', 'sum'), Passed=('Result', lambda x: (x == 'Pass').sum()))
score_by_task['Maximum'] = 10
score_by_task['Status'] = score_by_task['Passed'].map({0: 'Review', 1: 'Partially verified', 2: 'Checks passed'})

print('AUTOMATED SELF-ASSESSMENT · NOT FINAL HUMAN GRADE')
display(score_by_task[['Milestone', 'Points', 'Maximum', 'Status']])
print(f'Automated checks score: {assessment_df.Points.sum()}/100')
print('Checks needing attention:')
display(assessment_df.loc[assessment_df.Result == 'Review', ['Milestone', 'Check', 'Feedback']].reset_index(drop=True))

# Machine-readable output for independent, fresh-kernel instructor grading.
import json as _grade_json
print('SWIGGY_ASSESSMENT_JSON=' + _grade_json.dumps({
    'points': int(assessment_df.Points.sum()),
    'milestones': score_by_task[['Milestone', 'Points', 'Maximum', 'Status']].to_dict('records'),
    'checks': assessment_df[['Milestone', 'Check', 'Result', 'Points']].to_dict('records')
}))


AUTOMATED SELF-ASSESSMENT · NOT FINAL HUMAN GRADE


,Milestone,Points,Maximum,Status
0,1,10,10,Checks passed
1,2,10,10,Checks passed
2,3,10,10,Checks passed
3,4,10,10,Checks passed
4,5,10,10,Checks passed
5,6,10,10,Checks passed
6,7,10,10,Checks passed
7,8,10,10,Checks passed
8,9,10,10,Checks passed
9,10,10,10,Checks passed


Automated checks score: 100/100
Checks needing attention:


,Milestone,Check,Feedback


SWIGGY_ASSESSMENT_JSON={"points": 100, "milestones": [{"Milestone": 1, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 2, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 3, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 4, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 5, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 6, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 7, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 8, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 9, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 10, "Points": 10, "Maximum": 10, "Status": "Checks passed"}], "checks": [{"Milestone": 1, "Check": "Orders loaded, key fields present", "Result": "Pass", "Points": 5}, {"Milestone": 1, "Check": "Orders source is non-empty", "Result": "Pass", "Points": 5}, {"Mileston

### Submission handover
1. Save your completed Colab copy in Drive and share its link with the required permissions, or submit the `.ipynb` file if instructed by your instructor.
2. Include the final automated scorecard **and** your written findings in the notebook.
3. Do not submit the original three input datasets unless requested; output CSV and Excel files are generated during your run.
